# Huấn luyện PhoBERT / ViSoBERT trên Colab (Task 9)

Runtime → Change runtime type → **T4 GPU**. Chạy lần lượt từ trên xuống. Mỗi cấu hình ghi `results/runs/*.json`; sao lưu lên Drive sau mỗi lần chạy (ô cuối). **Không** dùng `--eval-test` ở đây (Task 11 mới chạy Test, một lần).

## 1. Kiểm tra GPU

In [1]:
!nvidia-smi

Thu Nov 19 17:28:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/nlp-absa-visfd"
!mkdir -p {DRIVE}

Mounted at /content/drive


## 3. Lấy code

In [4]:
REPO_URL = "https://github.com/CungDoHaiPhong279/nlp-absa-visfd.git"
%cd /content
!git clone -q {REPO_URL} repo || (cd repo && git pull -q)
%cd /content/repo
!git log --oneline -1

/content
/content/repo
cbc6874 (HEAD -> main, origin/main, origin/HEAD) Được tạo bằng Colab


## 4. Cài thư viện

In [5]:
!pip install -q -e . -r requirements.txt
!pip freeze > requirements.lock.colab.txt

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for absa (pyproject.toml) ... done


## 5. Java (VnCoreNLP cần Java 17+)

In [6]:
!java -version || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless)
!java -version

[0.017s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.017s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04

## 6. Tải dữ liệu

In [7]:
!python scripts/download_data.py

Đã tải vào data/raw


## 7. Khôi phục cache tách từ từ Drive (nếu có)

`results/runs` đã nằm trong git (D13), không cần khôi phục.

In [8]:
!mkdir -p data/processed
!cp -n {DRIVE}/data_processed/* data/processed/ 2>/dev/null; ls data/processed | head

## 8. Chạy test

In [9]:
!pytest -q

........................................................................ [ 75%]
.......................                                                  [100%]
95 passed in 32.66s


## 9. Kiểm tra nạp được tokenizer

In [10]:
from transformers import AutoTokenizer

for name in ("vinai/phobert-base-v2", "uitnlp/visobert"):
    print(name, AutoTokenizer.from_pretrained(name).tokenize("pin trâu, camera đẹp"))

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

vinai/phobert-base-v2 ['pin', 'tr@@', 'â@@', 'u@@', ',', 'camera', 'đẹp']


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  471kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

uitnlp/visobert ['▁p', 'in', '▁tr', 'âu', ',', '▁c', 'am', 'era', '▁đ', 'ẹp']


## 10. Đo thời gian (Task 9 Step 8)

500 dòng, 1 epoch. Ghi thời gian và VRAM (lệnh `nvidia-smi` ngay sau) vào `docs/decisions.md`, rồi ước lượng một lần chạy đầy đủ. Hơn 60 phút/lần: giảm `max_length` trước, rồi mới tăng `batch_size` lên 32.

In [11]:
!time python scripts/run.py --config configs/phobert_joint.yaml --limit 500 --set model.epochs=1
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-19 17:33:23 INFO  WordSegmenter:24 - Loading Word Segmentation model

pytorch_model.bin: downloading bytes:  74% 400M/540M [00:02<00:00, 255MB/s, 34.6MB/s  ]
pytorch_model.bin: downloading bytes:  80% 434M/540M [00:02<00:00, 222MB/s, 37.4MB/s  ]
pytorch_model.bin: downloading bytes:  89% 479M/540M [00:02<00:00, 222MB/s, 40.5MB/s  ]
pytorch_model.bin: reconstructing file:  74% 402M/540M [00:03<00:00, 155MB/s, 24.7MB/s  ] 
pytorch_model.bin: downloading bytes:  95% 512M/540M [00:03<00:00, 136MB/s, 42.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 514M/514M [00:03<00:00, 135MB/s, 42.6MB/s  ]
pytorch_model.bin: reconstructing file: 100% 540M/540M [00:03<

## 11. Chạy đầy đủ, seed 42 (Task 9 Step 9)

Mỗi ô một cấu hình. Chạy xong ô nào thì chạy ô sao lưu (mục 12) ngay, phòng Colab ngắt.

In [ ]:
!python scripts/run.py --config configs/phobert_joint.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E6_phobert_split.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E4_phobert_weight.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E3_phobert_strip_emoji.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E5_phobert_star.yaml

In [ ]:
!python scripts/run.py --config configs/visobert_joint.yaml

## 12. Sao lưu lên Drive

Sau đó tải các file `results/runs/*.json` mới về máy, chạy `python scripts/make_tables.py` và mở PR.

In [ ]:
import datetime

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
!mkdir -p {DRIVE}/data_processed {DRIVE}/backups/{stamp} {DRIVE}/models
!cp -r results {DRIVE}/backups/{stamp}/
!cp -n data/processed/* {DRIVE}/data_processed/ 2>/dev/null
!cp -r models/final_* {DRIVE}/models/ 2>/dev/null; ls {DRIVE}/backups/{stamp}/results/runs | tail